# Chapter 8 &mdash; Reading Off the Lengths Accepted by Any DFA

**Concept 12 of the Chapter 8 decomposition:** *Reading Off the Lengths of Strings Accepted by Any DFA*

Map every symbol to one letter with `apply_h_dfa`, determinize and minimize; the lasso shows the lengths.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Reading-Off-Lengths/Concept-Reading-Off-Lengths.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Concept 11's trick generalises. Given **any** DFA, apply the homomorphism that maps
**every** symbol to a single letter, say `1`. The resulting machine accepts $1^n$
exactly when the original accepts *some* string of length $n$.

`apply_h_dfa(D, h)` does the mapping and returns an **NFA** (several symbols collapse
onto one, so determinism is lost). Determinize and minimize, and you get the **lasso**
whose stem and cycle are the bound $b$ and period $p$ of Concept 10.

So the length set of any regular language can be read straight off a picture &mdash;
Concept 10's theorem, made constructive.

## 2. Definitions

### The homomorphism, applied

In [ ]:
def length_machine(D, letter='1'):
    N = apply_h_dfa(D, lambda c: letter)
    return min_dfa(nfa2dfa(N))

def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

### Reference: the true length set

In [ ]:
def length_set(D, upto):
    # Same state-set walk as Chapter 8, Concept 10.
    sig = sorted(D["Sigma"])
    cur, out = {D["q0"]}, set()
    for n in range(upto + 1):
        if cur & D["F"]: out.add(n)
        cur = {step_dfa(D, q, a) for q in cur for a in sig}
    return out

## 3. Tests

The collapsed machine accepts $1^n$ exactly for the achievable lengths $n$.

In [ ]:
D = re_dfa("(0+1)*1(0+1)(0+1)")
L = length_machine(D)
true_set = length_set(D, 12)
from_machine = {n for n in range(13) if accepts_dfa(L, '1'*n)}
print("true length set    :", sorted(true_set))
print("from the collapsed :", sorted(from_machine))
assert true_set == from_machine

The collapsed machine is small &mdash; it is the **lasso** of the length set.

In [ ]:
print("original minimal |Q| : %d" % len(D["Q"]))
print("length machine   |Q| : %d" % len(L["Q"]))
print("alphabet of the length machine :", sorted(L["Sigma"]))
assert L["Sigma"] == {'1'}

`apply_h_dfa` returns an **NFA**, because the homomorphism is not injective.

In [ ]:
N = apply_h_dfa(D, lambda c: '1')
print("has a Q0 key (so it is an NFA)? ", 'Q0' in N)
assert 'Q0' in N
print("|Q0| =", len(N["Q0"]), "  two 0/1 edges have become two 1-edges from one state")

Reading $b$ and $p$ straight off the lasso.

In [ ]:
def lasso(D):
    seq, seen = [], {}
    q = D["q0"]
    for n in range(len(D["Q"]) + 2):
        if q in seen: return seen[q], n - seen[q]
        seen[q] = n; q = step_dfa(D, q, '1')
    return None

for r in ["(000)*", "(111+11111)*", "(0+1)*1(0+1)(0+1)", "0*1*"]:
    Lm = length_machine(re_dfa(r))
    print("%-22s length machine |Q| = %2d, (stem, cycle) = %s"
          % (r, len(Lm["Q"]), lasso(Lm)))

Cross-check against Concept 11: the stamp language gives $Fr$ again.

In [ ]:
Lm = length_machine(re_dfa("(111+11111)*"))
print("|Q| = %d, so Fr = %d  (Sylvester: %d)" % (len(Lm["Q"]), len(Lm["Q"]) - 2, 3*5-3-5))
assert len(Lm["Q"]) - 2 == 7

## 4. Animation

The length lasso for $(0+1)^*1(0+1)(0+1)$ &mdash; stem, then cycle.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(length_machine(re_dfa('(0+1)*1(0+1)(0+1)')), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Take a DFA of your own and read off its length set this way.
2. Why must the collapsed machine be nondeterministic in general?
3. How does the cycle length relate to the period $p$ of Concept 10?

In [ ]:
# Your work for the exercises above.